# Large cross-encoder (Business Entity Resolution), Kaggle

Fine-tunes `FacebookAI/xlm-roberta-large` (MIT license, 560M parameters; weights downloaded from
Hugging Face, only the provided challenge data used) as a pair classifier on `cel_train.parquet`:
labeled train pairs of S1 entities OUTSIDE the pair-model sample, plus confident pairs of the
countries without training data labeled by the pair model itself (never by hand). Then it scores
every hard pair of the pair model: `cel_oof` (sample, for stage 2), `cel_test`, `cel_oof_loco`.

* **Settings:** Accelerator **GPU T4 x2**, Internet **On**, Add Input: the dataset with the
  four `cel_*.parquet` files.
* **Run:** Save Version > **Save & Run All (Commit)**: runs in the background (~2.5 h).
* **Output** (Output tab of the saved version): `cel_scores_oof.parquet`, `cel_scores_test.parquet`,
  `cel_scores_oof_loco.parquet` -> `student_resource/work/ce/`.

In [ ]:
import glob, math, os, time
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score, log_loss
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup


def find(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    assert hits, f'{name} not found: Add Input > the dataset with the cel_*.parquet files'
    return hits[0]


OUT = '/kaggle/working'
MODEL_NAME = 'FacebookAI/xlm-roberta-large'   # MIT, 560M parameters
MAX_LEN = 96
BATCH, EVAL_BATCH = 32, 256
LR, EPOCHS, WARMUP = 1e-5, 1, 0.06
MAX_TRAIN_SECONDS = 2.5 * 3600   # stop training early if needed, so scoring still ends by morning
SEED = 42
assert torch.cuda.is_available(), 'No GPU: Settings > Accelerator > GPU T4 x2'
N_GPU = torch.cuda.device_count()
print('GPUs:', N_GPU, [torch.cuda.get_device_name(i) for i in range(N_GPU)])
for f in ('cel_train', 'cel_oof', 'cel_test', 'cel_oof_loco'):
    print(find(f + '.parquet'))
torch.manual_seed(SEED)
np.random.seed(SEED)
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
class Logit(torch.nn.Module):
    """The model's single logit as a plain tensor, so DataParallel can gather it."""
    def __init__(self, m):
        super().__init__()
        self.m = m

    def forward(self, input_ids, attention_mask):
        return self.m(input_ids=input_ids, attention_mask=attention_mask).logits.squeeze(-1)


def batches(df, bs, shuffle=False, seed=SEED):
    idx = np.arange(len(df))
    if shuffle:
        np.random.default_rng(seed).shuffle(idx)
    for s in range(0, len(idx), bs):
        b = df.iloc[idx[s:s + bs]]
        enc = tok(list(b.text_a), list(b.text_b), truncation='longest_first',
                  max_length=MAX_LEN, padding=True, return_tensors='pt')
        yield b, {k: enc[k].cuda() for k in ('input_ids', 'attention_mask')}


@torch.no_grad()
def logits_of(net, df):
    net.eval()
    out = []
    with torch.autocast('cuda', dtype=torch.float16):
        for _, enc in batches(df, EVAL_BATCH):
            out.append(net(**enc).float().cpu().numpy())
    return np.concatenate(out) if out else np.zeros(0, dtype=np.float32)


def report(name, val, score):
    hard = val.p.between(0.02, 0.98).values
    for part, m in (('all held-out pairs', np.ones(len(val), bool)), ('hard pairs only', hard)):
        yv = val.label.values[m]
        print(f'{name} | {part}: {m.sum()} pairs | AUC {roc_auc_score(yv, score[m]):.4f} | '
              f'log loss {log_loss(yv, np.clip(score[m], 1e-6, 1 - 1e-6)):.4f}', flush=True)

## 1. Fine-tune

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1).cuda()
net = torch.nn.DataParallel(Logit(model)) if N_GPU > 1 else Logit(model)
data = pd.read_parquet(find('cel_train.parquet'))
held = pd.util.hash_pandas_object(data.s1_id, index=False).values % 50 == 0
val, tr = data[held], data[~held]
print(f'train {len(tr)} pairs, validation {len(val)}, positive rate {tr.label.mean():.3f}')
report('pair model', val, val.p.values)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
steps = EPOCHS * math.ceil(len(tr) / BATCH)
sched = get_linear_schedule_with_warmup(opt, int(WARMUP * steps), steps)
scaler = torch.amp.GradScaler('cuda')
loss_fn = torch.nn.BCEWithLogitsLoss()
step, t0, stop = 0, time.time(), False
for ep in range(EPOCHS):
    net.train()
    for b, enc in batches(tr, BATCH, shuffle=True, seed=SEED + 10 + ep):
        y = torch.tensor(b.label.values, dtype=torch.float32, device='cuda')
        with torch.autocast('cuda', dtype=torch.float16):
            logits = net(**enc)
        loss = loss_fn(logits.float(), y)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        sched.step()
        step += 1
        if step % 500 == 0 or step == 20:
            print(f'step {step}/{steps}  loss {loss.item():.4f}  {time.time() - t0:.0f}s', flush=True)
        if time.time() - t0 > MAX_TRAIN_SECONDS:
            print(f'time limit reached: training stops at step {step}/{steps}', flush=True)
            stop = True
            break
    if stop:
        break
model.save_pretrained(f'{OUT}/model_large')
tok.save_pretrained(f'{OUT}/model_large')
report('large     ', val, 1 / (1 + np.exp(-logits_of(net, val))))

## 2. Score every hard pair

In [ ]:
for name in ('oof', 'test', 'oof_loco'):
    df = pd.read_parquet(find(f'cel_{name}.parquet'), columns=['s1_id', 'match_id', 'text_a', 'text_b'])
    order = np.argsort((df.text_a.str.len() + df.text_b.str.len()).values, kind='stable')
    df = df.iloc[order].reset_index(drop=True)
    parts, t0 = [], time.time()
    for c0 in range(0, len(df), 100_000):
        parts.append(logits_of(net, df.iloc[c0:c0 + 100_000]))
        print(f'{name}: {min(c0 + 100_000, len(df))}/{len(df)}  {time.time() - t0:.0f}s', flush=True)
    pd.DataFrame({'s1_id': df.s1_id.values, 'match_id': df.match_id.values,
                  'ce_logit': np.concatenate(parts)}).to_parquet(f'{OUT}/cel_scores_{name}.parquet')
    print('wrote', f'cel_scores_{name}.parquet', flush=True)
print('Done. Output tab: download cel_scores_oof/test/oof_loco.parquet into student_resource/work/ce/')